Import libraries

In [6]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
import ipywidgets as widgets
from ipywidgets import interact

Data loading & Understanding

In [3]:
# 1. Load the Iris dataset
iris = load_iris()
X = iris.data
features = iris.feature_names

In [4]:
import pandas as pd

PCA

In [7]:
# Get PCA components
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

In [9]:
# when & Why to use PCA?
# PCA is used for dimensionality reduction, which can help in visualizing high-dimensional data, improving computational efficiency, and reducing noise. It is particularly useful when you have a large number of features and want to identify the most important ones that capture the variance in the data. PCA can also help in mitigating multicollinearity by transforming correlated features into uncorrelated principal components.

In [10]:
X_pca

array([[-2.68412563,  0.31939725],
       [-2.71414169, -0.17700123],
       [-2.88899057, -0.14494943],
       [-2.74534286, -0.31829898],
       [-2.72871654,  0.32675451],
       [-2.28085963,  0.74133045],
       [-2.82053775, -0.08946138],
       [-2.62614497,  0.16338496],
       [-2.88638273, -0.57831175],
       [-2.6727558 , -0.11377425],
       [-2.50694709,  0.6450689 ],
       [-2.61275523,  0.01472994],
       [-2.78610927, -0.235112  ],
       [-3.22380374, -0.51139459],
       [-2.64475039,  1.17876464],
       [-2.38603903,  1.33806233],
       [-2.62352788,  0.81067951],
       [-2.64829671,  0.31184914],
       [-2.19982032,  0.87283904],
       [-2.5879864 ,  0.51356031],
       [-2.31025622,  0.39134594],
       [-2.54370523,  0.43299606],
       [-3.21593942,  0.13346807],
       [-2.30273318,  0.09870885],
       [-2.35575405, -0.03728186],
       [-2.50666891, -0.14601688],
       [-2.46882007,  0.13095149],
       [-2.56231991,  0.36771886],
       [-2.63953472,

In [12]:
X_row_2d = X_pca[:, 2:4]
row_x_label = features[2]
row_y_label = features[3]


# What is Inertia in terms of clustering?
# Inertia refers to the distance of each data point from its assigned cluster center. It is a measure of how well the data points are grouped together. The lower the inertia, the better the clustering.


Clustering + Plotting

In [15]:
# Apply K-Means clustering on raw data
kmeans_raw = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_raw_labels = kmeans_raw.fit_predict(X)

# Plot K-Means clustering on raw data
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
scatter1 = plt.scatter(X[:, 0], X[:, 1], c=kmeans_raw_labels, cmap='viridis', s=50, alpha=0.6)
plt.scatter(kmeans_raw.cluster_centers_[:, 0], kmeans_raw.cluster_centers_[:, 1], 
            c='red', marker='X', s=200, edgecolors='black', linewidths=2, label='Centroids')
plt.xlabel(features[0])
plt.ylabel(features[1])
plt.title('K-Means Clustering on Raw Data')
plt.colorbar(scatter1, label='Cluster')
plt.legend()
plt.grid(True, alpha=0.3)

# Apply DBSCAN clustering on raw data
dbscan_raw = DBSCAN(eps=0.5, min_samples=5)
dbscan_raw_labels = dbscan_raw.fit_predict(X)

plt.subplot(1, 2, 2)
scatter2 = plt.scatter(X[:, 0], X[:, 1], c=dbscan_raw_labels, cmap='viridis', s=50, alpha=0.6)
plt.xlabel(features[0])
plt.ylabel(features[1])
plt.title('DBSCAN Clustering on Raw Data')
plt.colorbar(scatter2, label='Cluster')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [13]:
# Apply K-Means clustering on PCA-transformed data
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans_labels = kmeans.fit_predict(X_pca)

# Plot K-Means clustering results
plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=kmeans_labels, cmap='viridis', s=50, alpha=0.6)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], 
            c='red', marker='X', s=200, edgecolors='black', linewidths=2, label='Centroids')
plt.xlabel('First Principal Component')
plt.ylabel('Second Principal Component')
plt.title('K-Means Clustering on PCA-Transformed Data')
plt.colorbar(scatter, label='Cluster')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Assingment 1

### Doing K-Means clustering from scratch without sklearn only numpy

In [16]:
import numpy as np
import matplotlib.pyplot as plt

class KMeansFromScratch:
    def __init__(self, k=3, max_iters=100, tol=1e-4):
        self.k = k
        self.max_iters = max_iters
        self.tol = tol
        self.centroids = None
        self.labels = None

    def fit(self, X):
        # 1. Initialize centroids randomly from the dataset
        n_samples, n_features = X.shape
        random_indices = np.random.choice(n_samples, self.k, replace=False)
        self.centroids = X[random_indices]

        for i in range(self.max_iters):
            # 2. Assign each point to the nearest centroid
            distances = self._compute_distances(X)
            self.labels = np.argmin(distances, axis=1)

            # 3. Update centroids
            new_centroids = np.array([X[self.labels == j].mean(axis=0) if np.any(self.labels == j) else self.centroids[j] 
                                     for j in range(self.k)])

            # 4. Check for convergence
            if np.all(np.abs(new_centroids - self.centroids) < self.tol):
                break
            
            self.centroids = new_centroids
        
        return self

    def _compute_distances(self, X):
        # Euclidean distance: sqrt(sum((x-y)^2))
        # Using broadcasting for efficiency
        # X: (n_samples, n_features)
        # centroids: (k, n_features)
        # result: (n_samples, k)
        return np.sqrt(((X[:, np.newaxis] - self.centroids)**2).sum(axis=2))

    def predict(self, X):
        distances = self._compute_distances(X)
        return np.argmin(distances, axis=1)

# Example Usage with Iris data (assuming X is available from previous cells)
if 'X' in globals():
    km = KMeansFromScratch(k=3)
    km.fit(X)
    print("Centroids:\n", km.centroids)
    print("Labels unique counts:", np.unique(km.labels, return_counts=True))
else:
    print("Variable 'X' not found in workspace.")


Centroids:
 [[5.9016129  2.7483871  4.39354839 1.43387097]
 [6.85       3.07368421 5.74210526 2.07105263]
 [5.006      3.428      1.462      0.246     ]]
Labels unique counts: (array([0, 1, 2]), array([62, 38, 50]))


In [17]:
def visualize_scratch_kmeans(X, km):
    # Reduced dimensionality for plotting (assuming features[0], features[1] for 2D)
    x_axis = X[:, 0]
    y_axis = X[:, 1]

    plt.figure(figsize=(10, 6))
    plt.scatter(x_axis, y_axis, c=km.labels, cmap='viridis', label='Data Points')
    plt.scatter(km.centroids[:, 0], km.centroids[:, 1], c='red', marker='X', s=200, label='Centroids')
    plt.title('K-Means Clustering From Scratch (Iris Data - Dim 1 vs Dim 2)')
    plt.xlabel('Sepal Length')
    plt.ylabel('Sepal Width')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

if 'X' in globals():
    visualize_scratch_kmeans(X, km)
